In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
df = pd.read_csv('D:\\Fraud Detection model project 2\\Raw_dataset\\financial_fraud_detection_cleaned.csv')

In [22]:
df.head()

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Amount,Merchant_Category,Payment_Method,Device_Type,Location,Is_International,Previous_Transactions,Average_Spend,Account_Age_Days,Suspicious_Keyword,Fraudulent
0,T100000,CUST3252,2023-10-04 07:45:00,37.54,Travel,PayPal,POS,Bengaluru,0,94,417.40,1492,No,0
1,T100001,CUST1630,2023-09-26 08:29:00,240.81,Utilities,PayPal,Mobile,Bengaluru,0,76,335.47,66,No,0
2,T100002,CUST7852,2023-07-18 15:54:00,105.34,Entertainment,Debit Card,Mobile,Bengaluru,0,60,432.03,216,No,0
3,T100003,CUST4892,2023-10-16 17:10:00,73.04,Utilities,NetBanking,Desktop,Kolkata,0,68,488.30,449,No,0
4,T100004,CUST8831,2023-09-09 00:46:00,13.57,Utilities,Credit Card,Desktop,Bengaluru,0,57,437.03,754,No,0


In [23]:
print(df["Transaction_Date"].dtype)
print(df["Transaction_Date"].head(10))

str
0    2023-10-04 07:45:00
1    2023-09-26 08:29:00
2    2023-07-18 15:54:00
3    2023-10-16 17:10:00
4    2023-09-09 00:46:00
5    2024-01-14 03:15:00
6    2023-02-09 08:56:00
7    2023-03-26 13:12:00
8    2023-03-06 20:09:00
9    2023-06-27 12:06:00
Name: Transaction_Date, dtype: str


In [24]:
df.shape

(5000, 14)

In [25]:
# checking date is in proper datetime format
df["Transaction_Date"] = pd.to_datetime(
    df["Transaction_Date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

print(df["Transaction_Date"].dtype)
print("Missing dates:", df["Transaction_Date"].isna().sum())

datetime64[us]
Missing dates: 0


In [26]:
df["Transaction_Hour"] = df["Transaction_Date"].dt.hour

print(
    df[
        ["Transaction_Date", "Transaction_Hour"]
    ].head(10)
)

     Transaction_Date  Transaction_Hour
0 2023-10-04 07:45:00                 7
1 2023-09-26 08:29:00                 8
2 2023-07-18 15:54:00                15
3 2023-10-16 17:10:00                17
4 2023-09-09 00:46:00                 0
5 2024-01-14 03:15:00                 3
6 2023-02-09 08:56:00                 8
7 2023-03-26 13:12:00                13
8 2023-03-06 20:09:00                20
9 2023-06-27 12:06:00                12


In [27]:
df["Day_of_Week"] = df["Transaction_Date"].dt.dayofweek

print(
     df[["Transaction_Date", "Day_of_Week"]].head(10)
)

     Transaction_Date  Day_of_Week
0 2023-10-04 07:45:00            2
1 2023-09-26 08:29:00            1
2 2023-07-18 15:54:00            1
3 2023-10-16 17:10:00            0
4 2023-09-09 00:46:00            5
5 2024-01-14 03:15:00            6
6 2023-02-09 08:56:00            3
7 2023-03-26 13:12:00            6
8 2023-03-06 20:09:00            0
9 2023-06-27 12:06:00            1


In [28]:
df["Transaction_Month"] = df["Transaction_Date"].dt.month

print(
    df[["Transaction_Date", "Transaction_Month"]].head(10)
)

     Transaction_Date  Transaction_Month
0 2023-10-04 07:45:00                 10
1 2023-09-26 08:29:00                  9
2 2023-07-18 15:54:00                  7
3 2023-10-16 17:10:00                 10
4 2023-09-09 00:46:00                  9
5 2024-01-14 03:15:00                  1
6 2023-02-09 08:56:00                  2
7 2023-03-26 13:12:00                  3
8 2023-03-06 20:09:00                  3
9 2023-06-27 12:06:00                  6


In [29]:
df["Is_Weekend"] = (
    df["Day_of_Week"] >=5
).astype(int)

print(
    df[["Day_of_Week", "Is_Weekend"]].head(10)
)

   Day_of_Week  Is_Weekend
0            2           0
1            1           0
2            1           0
3            0           0
4            5           1
5            6           1
6            3           0
7            6           1
8            0           0
9            1           0


In [30]:
# High Transaction in Amount
amount_threshold = df["Transaction_Amount"].quantile(0.75)

print("High amount threshold:", amount_threshold)

df["High_Amount"] = (
    df["Transaction_Amount"] > amount_threshold
).astype(int)

print(
    df[
        ["Transaction_Amount", "High_Amount"]
    ].head(10)
)

High amount threshold: 110.2975
   Transaction_Amount  High_Amount
0               37.54            0
1              240.81            1
2              105.34            0
3               73.04            0
4               13.57            0
5               13.57            0
6                4.79            0
7              160.90            1
8               73.53            0
9               98.50            0


In [31]:
# Transaction based on night time
df["Is_Night"] = (
    (df["Transaction_Hour"] >= 22) |
    (df["Transaction_Hour"] <= 5)
).astype(int)

print(
    df[
        ["Transaction_Hour", "Is_Night"]
    ].head(20)
)

    Transaction_Hour  Is_Night
0                  7         0
1                  8         0
2                 15         0
3                 17         0
4                  0         1
5                  3         1
6                  8         0
7                 13         0
8                 20         0
9                 12         0
10                22         1
11                18         0
12                 4         1
13                21         0
14                10         0
15                12         0
16                 5         1
17                12         0
18                 9         0
19                15         0


In [32]:
# Review all engineering features

engineered_features = [
    "Transaction_Hour",
    "Day_of_Week",
    "Transaction_Month",
    "Is_Weekend",
    "High_Amount",
    "Is_Night"
]

print(df[engineered_features].head(10))

   Transaction_Hour  Day_of_Week  Transaction_Month  Is_Weekend  High_Amount  \
0                 7            2                 10           0            0   
1                 8            1                  9           0            1   
2                15            1                  7           0            0   
3                17            0                 10           0            0   
4                 0            5                  9           1            0   
5                 3            6                  1           1            0   
6                 8            3                  2           0            0   
7                13            6                  3           1            1   
8                20            0                  3           0            0   
9                12            1                  6           0            0   

   Is_Night  
0         0  
1         0  
2         0  
3         0  
4         1  
5         1  
6         0  
7      

In [33]:
print(df[engineered_features].isna().sum())

Transaction_Hour     0
Day_of_Week          0
Transaction_Month    0
Is_Weekend           0
High_Amount          0
Is_Night             0
dtype: int64


In [34]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 5000
Columns: 20


In [35]:
df.head()

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Transaction_ID         5000 non-null   str           
 1   Customer_ID            5000 non-null   str           
 2   Transaction_Date       5000 non-null   datetime64[us]
 3   Transaction_Amount     5000 non-null   float64       
 4   Merchant_Category      5000 non-null   str           
 5   Payment_Method         5000 non-null   str           
 6   Device_Type            5000 non-null   str           
 7   Location               5000 non-null   str           
 8   Is_International       5000 non-null   int64         
 9   Previous_Transactions  5000 non-null   int64         
 10  Average_Spend          5000 non-null   float64       
 11  Account_Age_Days       5000 non-null   int64         
 12  Suspicious_Keyword     5000 non-null   str           
 13  Fraudulent    

In [37]:
# save the feature-engineered dataset file.
df.to_csv("../Raw_dataset/financial_fraud_detection_feature_engineered.csv", index=False)

print("created sucessfully")

created sucessfully
